# NumPy и векторные вычисления

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи самостоятельно, затем обратитесь к разделу решений.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Массивы и оси NumPy](../../python/numerical-data/numpy-arrays/)
- [Индексация, broadcasting и векторизация](../../python/numerical-data/numpy-indexing/)


## Чему научимся

Хранить числовую таблицу массивом, выбирать ось вычисления и проверять формы до операции. Нужны коллекции, векторы и матрицы. Для запуска установите NumPy по инструкции во введении.

## Массив, форма и тип

NumPy — библиотека числовых вычислений. Запись `import numpy as np` подключает её под коротким именем `np`; вызовы начинаются с этого имени. `np.array` создаёт массив из списка или вложенных списков. Рассматриваем числовые массивы: все их элементы имеют общий тип.

Свойство `shape` — кортеж длин осей, `ndim` — число осей, `dtype` — тип элемента, `nbytes` — число байт элементов без служебных данных. У таблицы ось 0 перечисляет строки, ось 1 — столбцы. Форма `(3, 2)` означает три строки и два столбца; `(3,)` — одну ось длины 3. Запятая в `(3,)` нужна для кортежа с одним элементом.

Тип `np.float64` занимает 8 байт на число и хранит дробные значения приближённо. Целые типы вроде `int8` имеют ограниченный диапазон: слишком большой результат может переполниться. Это отличается от обычных целых Python. Выбирайте тип по диапазону, а не только ради экономии памяти.

In [ ]:
import numpy as np

X = np.array([[12, 8], [22, 10], [14, 9]], dtype=np.float64)
assert X.shape == (3, 2)
assert X.ndim == 2
assert X.nbytes == 3 * 2 * 8


Строки — три вымышленных объекта, столбцы — размах в сантиметрах и масса в граммах. Например, `X[0, 1]` равно 8: строка 0, столбец 1.

## Среднее и сумма по осям

Методы `mean` и `sum` вычисляют среднее и сумму. Параметр `axis` указывает, какую ось убрать вычислением: `axis=0` объединяет строки и оставляет результат для каждого столбца, `axis=1` объединяет столбцы и оставляет результат для каждой строки. Без `axis` операция использует все элементы.

In [ ]:
print(X.mean(axis=0))  # [16.  9.]
print(X.sum(axis=1))   # [20. 32. 23.]


Первое среднее равно `(12 + 22 + 14) / 3 = 16`. Вторая строка вывода содержит суммы размаха и массы; это лишь демонстрация оси, у такой суммы разных единиц нет полезного физического смысла. Для пустого набора среднее не определено; проверяйте число строк заранее.

## Индексация, маски и копии

В `X[строки, столбцы]` двоеточие выбирает всю ось. Логическая маска хранит `True` для нужных строк. Маски объединяют `&`, `|`, `~` со скобками: «и», «или», «не». Например, `(X[:, 0] >= 15) & (X[:, 1] < 11)`. Операторы `and` и `or` для целых массивов не подходят: массив содержит много логических значений.

Обычный срез массива часто даёт **представление**: оно использует ту же память, что и исходный массив. Изменение элементов представления меняет исходный массив. Выбор логической маской создаёт копию. Для независимости явно используйте `.copy()`. Функция `np.shares_memory(a, b)` проверяет, есть ли у массивов общая память.

In [ ]:
print(X[:, 0])             # [12. 22. 14.]
mask = X[:, 0] >= 15
selected = X[mask]
assert selected.shape == (1, 2)
view = X[:1]
independent = X[:1].copy()
assert np.shares_memory(view, X)
assert not np.shares_memory(independent, X)


## Broadcasting: согласование форм

При поэлементной операции NumPy сравнивает размеры справа налево. Размеры совместимы, если равны или один из них равен 1; отсутствующие слева оси считаются единичными. Такое согласование называют broadcasting. Например, `(3, 2)` и `(2,)` согласуются как `(3, 2)` и `(1, 2)`: один вектор вычитается из каждой строки.

Метод `reshape` меняет форму при сохранении числа элементов. `np.allclose` проверяет приблизительное равенство всех чисел с абсолютным и относительным допуском: допускаются малые различия и учитывается масштаб сравниваемых значений. Для точного сравнения массивов одинаковой формы есть `np.array_equal`.

In [ ]:
center = X.mean(axis=0)
centered = X - center
assert centered.shape == (3, 2)
assert np.allclose(centered.mean(axis=0), [0, 0])
a = np.array([1., 2., 3.])
column = a.reshape(3, 1)
assert (column - a).shape == (3, 3)


Последняя операция — частая ошибка: `(3, 1)` и `(3,)` согласуются как `(3, 1)` и `(1, 3)`. Получаются все девять разностей, а не три разности соответствующих элементов. Перед операцией полезно выписать формы на бумаге.

При подготовке модели среднее вычисляют только на обучающих строках, затем вычитают **то же** среднее из проверочных. Если сначала вычислить среднее всей таблицы, подготовка использует информацию о проверочной части: это утечка данных. Центрирование выше показывает арифметику; для оценки модели таблицу сначала разделяют.

## Разобранный пример: расстояния без цикла Python

Вычтем координаты запроса из каждой строки, возведём разности в квадрат и сложим по координатам. `np.argmin` даёт индекс первого наименьшего значения. Квадрата расстояния достаточно для сравнения соседей; если нужна сама длина, `np.sqrt` извлечёт квадратный корень каждого элемента. Как в математике, `@` умножает матрицу на вектор, а `*` — элементы.

In [ ]:
query = np.array([15., 9.])
differences = X - query
squared_distances = (differences ** 2).sum(axis=1)
assert np.allclose(squared_distances, [10, 50, 1])
assert int(np.argmin(squared_distances)) == 2
weights = np.array([2., -1.])
assert np.allclose(X @ weights, [16, 34, 19])


Для первой строки получаем $(12-15)^2+(8-9)^2=10$. Это **векторизация**: работа передана библиотечным операциям. Она не отменяет расход памяти — здесь создаётся таблица разностей. Единицы признаков всё ещё влияют на результат, как в главе о расстояниях.

## Случайность и точность

`np.random.default_rng(seed)` создаёт генератор случайных чисел с заданным начальным состоянием — seed. Метод `random` выдаёт числа от 0 включительно до 1 не включительно в массиве указанной формы. `np.isclose` сравнивает отдельные числа с допуском.

In [ ]:
rng = np.random.default_rng(17)
noise = rng.random(size=(3, 2))
assert noise.shape == X.shape
assert np.isclose(0.1 + 0.2, 0.3)


Повторный вызов того же генератора выдаст следующие числа; для повторения создайте новый генератор с тем же seed в том же окружении. Дробные вычисления сравнивайте с допуском: точное равенство `0.1 + 0.2 == 0.3` обычно ложно.

Для справки: [руководство NumPy](https://numpy.org/doc/stable/user/quickstart.html).

## Задачи

### 1. Базовая

Отдельный массив `A` имеет форму `(5, 3)`. Какую форму имеет результат `A.mean(axis=1)`?

<details>
<summary>Подсказка</summary>

Сворачиваются столбцы.

</details>



### 2. Базовая

Сколько байт занимают элементы массива (100, 4) типа float64?

<details>
<summary>Подсказка</summary>

Один элемент занимает 8 байт.

</details>



### 3. Прикладная

Выберите из таблицы `X` разобранного примера строки с размахом не меньше 14 и массой не больше 9.

<details>
<summary>Подсказка</summary>

Используйте две маски со скобками.

</details>



### 4. Прикладная

Вычислите длины векторов (3, 4) и (5, 12) без цикла.

<details>
<summary>Подсказка</summary>

Суммируйте квадраты по координатам, затем извлеките корень.

</details>



### 5. Олимпиадная

Ответы имеют форму (100,), предсказания — (100, 1). Почему средний квадрат разности может вычислиться без ошибки, но неверно?

<details>
<summary>Подсказка</summary>

Проверьте форму разности.

</details>



### 6. Олимпиадная

Почему вычитание среднего всей таблицы перед её разбиением портит проверку модели?

<details>
<summary>Подсказка</summary>

Среднее несёт информацию и о будущей проверочной части.

</details>



## Итоги и самопроверка

Перед выполнением предскажите формы результатов. Умейте объяснить ось среднего, правила broadcasting и отличие среза от копии. Скорость векторизации не освобождает от проверки памяти и смысла признаков.


## Решения задач

### Задача 1

(5,).

### Задача 2

3200 байт, без служебного объекта массива.

### Задача 3

In [ ]:
chosen = X[(X[:, 0] >= 14) & (X[:, 1] <= 9)]
assert np.array_equal(chosen, [[14, 9]])


### Задача 4

In [ ]:
vectors = np.array([[3., 4.], [5., 12.]])
lengths = np.sqrt((vectors ** 2).sum(axis=1))
assert np.allclose(lengths, [5, 13])


### Задача 5

Broadcasting создаст (100, 100): сравнятся все пары. Нужно заранее согласовать формы, например привести обе к (100,), и проверить их равенство.

### Задача 6

Подготовка уже использовала распределение проверочных данных. Среднее нужно вычислять только по обучающей части, а затем применять к остальным без пересчёта.